In [1]:

import warnings
warnings.filterwarnings('ignore')

import gc
import torch
gc.collect()
torch.cuda.empty_cache()

#Ramanujan1729,push

# Loading Libraries
import pandas as pd
import numpy as np
import pickle
import os
import copy
import time

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, cohen_kappa_score




# Input Parameters

In [2]:
# File paths and other parameters
Nbins= 2
folder = 'Feature_Importance'
property= 'Activity'
target_style= 'NG'
featurizer_style= 'All_Featurizers_'+target_style
featurizer_name = 'RDKit_Descriptors_NG'
sheet_name = featurizer_name


feature_list_dict= {'First 3':[1, 3, 11, 14, 22, 24, 38, 46, 70, 841],
                    'First 5':[0, 1, 3, 11, 14, 22, 23, 24, 25, 38, 44, 46, 63, 70, 106, 122, 251, 647, 654, 841, 842],
                    'First 7':[0, 1, 3, 11, 14, 20, 22, 23, 24, 25, 30, 38, 40, 44, 46, 53, 57, 59, 60, 63, 65, 66, 70, 73, 98, 106, 115, 122, 140, 224, 238, 251, 257, 305, 421, 423, 647, 648, 654, 841, 842],
                    'First 10':[0, 1, 3, 11, 13, 14, 19, 20, 21, 22, 23, 24, 25, 29, 30, 31, 38, 40, 43, 44, 45, 46, 51, 53, 57, 59, 60, 63, 65, 66, 70, 73, 83, 86, 93, 96, 98, 103, 106, 107, 115, 116, 117, 122, 135, 140, 217, 224, 227, 238, 240, 251, 256, 257, 305, 310, 311, 344, 421, 423, 483, 647, 648, 654, 716, 841, 842],
                    'Second 3':[23, 24, 25, 44, 63, 66, 421, 423, 654, 842],
                    'Second 5':[11, 13, 14, 22, 24, 30, 38, 43, 45, 53, 57, 59, 65, 66, 73, 93, 96, 116, 421, 423, 483],
                    'Second 7':[0, 1, 3, 11, 13, 14, 15, 19, 22, 23, 24, 38, 43, 45, 51, 53, 57, 59, 63, 65, 66, 67, 70, 73, 77, 86, 93, 96, 101, 107, 116, 122, 135, 250, 483, 668, 841, 842],
                    'Second 10':[0, 1, 3, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 34, 35, 43, 44, 46, 47, 52, 53, 57, 59, 63, 65, 66, 67, 74, 76, 77, 82, 83, 86, 94, 95, 96, 98, 99, 101, 107, 115, 116, 122, 139, 198, 218, 250, 494, 668, 692, 837, 841, 842],
                    'Third 3':[30, 43, 45, 53, 59, 65, 73, 116, 483],
                    'Third 5':[0, 1, 3, 11, 14, 15, 20, 22, 23, 24, 47, 63, 65, 66, 67, 77, 86, 107, 116, 122, 668, 841, 842],
                    'Third 7':[0, 11, 12, 14, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 34, 43, 44, 47, 53, 59, 61, 65, 66, 74, 76, 77, 82, 83, 86, 94, 96, 115, 122, 139, 494, 525, 837],
                    'Third 10':[0, 1, 2, 3, 6, 8, 11, 12, 14, 15, 18, 20, 22, 23, 24, 25, 28, 29, 31, 34, 35, 40, 41, 42, 43, 44, 46, 47, 50, 51, 53, 57, 61, 63, 64, 65, 66, 70, 73, 74, 76, 77, 82, 83, 86, 91, 94, 96, 97, 99, 101, 104, 106, 115, 116, 117, 140, 191, 198, 207, 217, 239, 443, 483, 525, 538, 650, 841, 842],
                    'Fifth 3':[11, 14, 20, 22, 24, 47, 66, 67, 77, 86, 841],
                    'Fifth 5':[0, 3, 12, 18, 20, 23, 25, 43, 53, 57, 61, 63, 65, 70, 74, 82, 83, 86, 96, 104, 106, 117, 191, 217, 239, 525, 538],
                    'Fifth 7':[0, 3, 8, 9, 11, 13, 14, 18, 20, 21, 22, 23, 24, 25, 26, 31, 33, 41, 43, 44, 45, 46, 50, 57, 59, 64, 66, 73, 74, 83, 92, 93, 99, 104, 139, 192, 198, 207, 344, 443, 483, 841, 842],
                    'Fifth 10':[0, 1, 3, 4, 7, 8, 11, 13, 14, 15, 16, 18, 19, 20, 22, 24, 25, 26, 29, 30, 37, 42, 43, 44, 45, 46, 53, 56, 59, 63, 65, 66, 67, 74, 75, 78, 83, 89, 94, 99, 103, 104, 105, 106, 114, 115, 116, 117, 122, 140, 176, 192, 198, 207, 218, 303, 305, 344, 438, 443, 716, 841, 842],
                    'Tenth 3':[11, 14, 18, 25, 35, 41, 44, 46, 64, 82, 115, 841],
                    'Tenth 5':[0, 1, 3, 7, 11, 14, 19, 22, 24, 30, 37, 43, 63, 65, 78, 83, 94, 99, 114, 115, 116, 122, 192, 218, 305, 438, 842],
                    'Tenth 7':[0, 1, 3, 6, 8, 11, 12, 14, 18, 20, 23, 24, 25, 32, 35, 38, 44, 47, 57, 66, 70, 82, 87, 88, 96, 103, 104, 107, 116, 123, 140, 158, 191, 225, 251, 295, 343, 422, 841],
                    'Tenth 10':[0, 1, 2, 3, 4, 6, 11, 13, 14, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 29, 30, 31, 34, 36, 37, 38, 42, 44, 45, 46, 47, 51, 53, 57, 59, 63, 65, 66, 67, 73, 75, 76, 77, 78, 82, 83, 84, 86, 90, 96, 98, 99, 104, 105, 115, 126, 134, 139, 164, 191, 761, 841, 842],
                    '15th 3':[18, 24, 25, 30, 67, 103, 104, 105, 106, 841, 842],
                    '15th 5':[6, 11, 14, 19, 20, 22, 30, 40, 45, 53, 60, 63, 70, 74, 83, 86, 93, 94, 103, 106, 107, 117, 139, 191, 235, 272],
                    '15th 7':[1, 11, 14, 15, 17, 18, 19, 20, 21, 24, 25, 27, 29, 40, 44, 45, 53, 59, 63, 66, 73, 76, 77, 83, 86, 88, 89, 104, 105, 114, 140, 230, 234, 311, 373, 447, 750],
                    '15th 10':[0, 1, 2, 3, 6, 7, 8, 11, 14, 15, 16, 18, 20, 21, 22, 23, 24, 25, 26, 29, 33, 35, 36, 38, 40, 42, 43, 45, 47, 51, 53, 56, 57, 59, 60, 66, 70, 73, 74, 82, 84, 86, 92, 93, 94, 96, 99, 102, 104, 105, 109, 115, 122, 123, 135, 139, 192, 199, 207, 217, 223, 252, 261, 265, 274, 279, 297, 302, 310, 323, 343, 344, 422, 666, 841, 842],
                    '20th 3':[6, 11, 14, 18, 22, 45, 53, 59, 66, 67, 106, 236],
                    '20th 5':[11, 14, 17, 18, 20, 21, 24, 26, 51, 59, 63, 66, 73, 76, 82, 86, 98, 99, 115, 761],
                    '20th 7':[0, 1, 3, 4, 6, 13, 14, 15, 20, 21, 22, 23, 27, 31, 38, 43, 44, 46, 47, 48, 50, 53, 61, 63, 64, 65, 66, 74, 82, 83, 87, 88, 91, 94, 96, 97, 98, 99, 100, 104, 114, 139, 164, 198, 207, 230, 234, 257, 311, 331, 538],
                    '20th 10':[0, 1, 4, 5, 6, 8, 12, 13, 14, 18, 19, 20, 23, 24, 28, 30, 31, 32, 33, 37, 48, 53, 61, 65, 74, 83, 84, 87, 96, 97, 99, 105, 106, 107, 110, 116, 117, 122, 125, 126, 175, 207, 210, 225, 251, 301, 308, 316, 344, 374, 450, 459, 643, 842]
                    }


dataX_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\dataX_dict_all_{featurizer_style}.pkl"
datay_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\datay_all_{Nbins}bins_{target_style}.pkl"
scaffolds_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\Scaffolds.pkl"

metrics_output_path = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{folder}\{property}_Metrics {Nbins} bins.xlsx"



# Load datasets

In [3]:
# Load datasets

with open(dataX_filepath, 'rb') as file:
    print('Reading feature variable')
    dataX_dict_all = pickle.load(file)
    
with open(datay_filepath, 'rb') as file1:
    print('Reading target variable')
    datay_all= pickle.load(file1)

with open(scaffolds_filepath, 'rb') as file1:
    print('Reading scaffolds file')
    scaffolds= pickle.load(file1)

features_full= dataX_dict_all[featurizer_name]

Reading feature variable
Reading target variable
Reading scaffolds file


# Save Function

In [4]:

def save_metrics(datay_dict_pred, datay_test, classes, fold_idx, output_file):
    rows = []
    results_sheet_name= f'Fold {fold_idx + 1}'
    for feature_list_name, predictions in datay_dict_pred.items():
        row = {'Feature List Name': feature_list_name}
        for cls in classes:
            y_true_binary = [1 if y == cls else 0 for y in datay_test]
            y_pred_binary = [1 if y == cls else 0 for y in predictions]
            row[f'Accuracy_{cls}'] = accuracy_score(y_true_binary, y_pred_binary)
            row[f'Precision_{cls}'] = precision_score(y_true_binary, y_pred_binary)
            row[f'Recall_{cls}'] = recall_score(y_true_binary, y_pred_binary)
            row[f'F1 Score_{cls}'] = f1_score(y_true_binary, y_pred_binary)
            row[f'MCC_{cls}'] = matthews_corrcoef(y_true_binary, y_pred_binary)
            row[f'AUC_{cls}'] = roc_auc_score(y_true_binary, y_pred_binary) if len(np.unique(y_true_binary)) > 1 else np.nan
        row['Kappa'] = cohen_kappa_score(datay_test, predictions)
        rows.append(row)

    results_df = pd.DataFrame(rows)
    
    if os.path.exists(output_file):
        with pd.ExcelFile(output_file, engine='openpyxl') as xls:
            if results_sheet_name in xls.sheet_names:
                existing_data_df = pd.read_excel(xls, sheet_name=results_sheet_name)
                # Concatenate existing data with new data
                combined_df = pd.concat([existing_data_df, results_df], ignore_index=True)
            else:
                combined_df = results_df
    else:
        combined_df = results_df
    
    # Write the combined DataFrame to the Excel file
    with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        combined_df.to_excel(writer, sheet_name=results_sheet_name, index=False)
    
    print(f"Finished")





# 5 Fold CV

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=4, min_samples_split=10, random_state=42, class_weight='balanced')


for fold_idx, (train_idx, test_idx) in enumerate(kf.split(scaffolds)):
    print(f"\nFold {fold_idx + 1}")
    datay_train, datay_test = datay_all[train_idx], datay_all[test_idx]

    datay_dict_pred = {}
    for feature_list_name, feature_list in feature_list_dict.items():
        
        features = features_full[:, feature_list]
        features_train, features_test = features[train_idx], features[test_idx]
        
        model.fit(features_train, datay_train)
        datay_dict_pred[feature_list_name] = model.predict(features_test)

    save_metrics(datay_dict_pred, datay_test, np.unique(datay_all), fold_idx, metrics_output_path)

print("\n✅ Scaffold-Based Cross-Validation Completed and All Outputs Saved.")




Fold 1
Finished

Fold 2
Finished

Fold 3
Finished

Fold 4
Finished

Fold 5
Finished

✅ Scaffold-Based Cross-Validation Completed and All Outputs Saved.
